#  Database Schema Design (DuckDB)

## Overview

In this project, we design a multi-layer analytical database using DuckDB. The goal is to structure our weather data pipeline into clear stages that improve data reliability, maintainability, and analytical usability.

We define three main layers:

* **raw** → original ingested data (no changes)
* **staging** → cleaned and validated data
* **analytics** → feature-engineered data for modeling

---

#  1. RAW Layer

## Table: `raw_weather_daily`

### Purpose

This table stores the original ingested weather data exactly as retrieved from the API. No transformations or cleaning are applied.

### Columns

| Column                     | Data Type | Description                       |
| -------------------------- | --------- | --------------------------------- |
| date                       | TIMESTAMP | Date of observation               |
| city                       | VARCHAR   | City name                         |
| weathercode                | FLOAT     | Weather condition code            |
| temperature_2m_max         | FLOAT     | Maximum daily temperature (°C)    |
| temperature_2m_min         | FLOAT     | Minimum daily temperature (°C)    |
| apparent_temperature_max   | FLOAT     | Max perceived temperature         |
| apparent_temperature_min   | FLOAT     | Min perceived temperature         |
| precipitation_sum          | FLOAT     | Total precipitation (mm)          |
| precipitation_hours        | FLOAT     | Hours of precipitation            |
| rain_sum                   | FLOAT     | Rainfall amount (mm)              |
| snowfall_sum               | FLOAT     | Snowfall amount (cm/mm)           |
| windspeed_10m_max          | FLOAT     | Max wind speed (km/h)             |
| windgusts_10m_max          | FLOAT     | Max wind gusts (km/h)             |
| winddirection_10m_dominant | FLOAT     | Dominant wind direction (degrees) |

---

#  2. STAGING Layer

## Table: `staging_weather_daily`

### Purpose

This layer contains cleaned and validated data. It ensures:

* No missing values
* Correct data types
* No duplicate rows

### Transformations

* Null values handled (if any)
* Duplicate rows removed
* Data types enforced

### Columns

(Same as raw layer, but cleaned)

---

#  3. ANALYTICS Layer

## Table: `analytics_weather_features`

### Purpose

This table is used for analysis and machine learning. It contains derived features based on raw weather data.

### Example Features

* Rolling averages (7-day temperature mean)
* Seasonal indicators (sin/cos transformations)
* Lag features (previous day values)

### Columns (Example)

| Column               | Description                       |
| -------------------- | --------------------------------- |
| temp_7d_avg          | 7-day rolling average temperature |
| precipitation_7d_sum | 7-day total precipitation         |
| wind_avg_7d          | Average wind speed (7 days)       |
| day_of_year_sin      | Seasonal encoding                 |
| day_of_year_cos      | Seasonal encoding                 |

---

#  Design Rationale

* **Raw layer** ensures data traceability
* **Staging layer** ensures data quality
* **Analytics layer** enables efficient modeling and analysis

This layered approach follows best practices in modern data engineering pipelines.


## 1. Connect + Load setup

In [3]:
import duckdb
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath("../src")) 

from database import (
    get_connection,
    create_schemas,
    create_raw_tables,
    load_raw_historical_data,
    load_raw_forecast_data
)

# =========================
# DB PATH
# =========================
db_path = "../data/database/weather_daily.duckdb"

os.makedirs(os.path.dirname(db_path), exist_ok=True)

# =========================
# CONNECTION
# =========================
conn = get_connection(db_path)

# =========================
# SCHEMAS
# =========================
create_schemas(conn)

# =========================
# CLEAN TABLES (SAFE RESET)
# =========================
conn.execute("DROP TABLE IF EXISTS raw.weather_daily_historical;")
conn.execute("DROP TABLE IF EXISTS raw.weather_daily_forecast;")

# =========================
# TABLES (RECREATE)
# =========================
create_raw_tables(conn)

# =========================
# LOAD HISTORICAL DATA
# =========================
load_raw_historical_data(conn, "../data/raw")

# =========================
# LOAD FORECAST DATA
# =========================
load_raw_forecast_data(conn, "../data/raw")

Loading Historical CSV: Ganja_historical.csv
Loading Historical CSV: Nakhchivan_historical.csv
Loading Historical CSV: Baku_historical.csv
Loading Historical CSV: Shusha_historical.csv
Loading Forecast CSV: Baku_forecast.csv
Loading Forecast CSV: Ganja_forecast.csv
Loading Forecast CSV: Nakhchivan_forecast.csv
Loading Forecast CSV: Shusha_forecast.csv


##  Database Setup & Data Loading Pipeline

In this step, we initialize the local analytical database using **DuckDB** and load both historical and forecast weather data into separate raw-layer tables.

---

###  1. Database Connection
We create or connect to a local DuckDB database file:

- `get_connection(db_path)` → Opens or creates the database file (`weather_daily.duckdb`)

---

###  2. Schema Initialization
We define a multi-layer database architecture:

- **raw** → Stores unprocessed ingested data  
- **staging** → Cleaned and validated data (future use)  
- **analytics** → Feature-engineered data for analysis and ML  

Function used:
- `create_schemas(conn)` → Creates schemas if they do not exist

---

###  3. Table Reset (Clean State)
Before each pipeline run, we remove old data to ensure reproducibility:

- `DROP TABLE IF EXISTS raw.weather_daily_historical`
- `DROP TABLE IF EXISTS raw.weather_daily_forecast`

This ensures the pipeline is **idempotent** (same output on every run).

---

###  4. Raw Table Creation
We create two raw tables with identical schema:

- `raw.weather_daily_historical` → training dataset (historical weather data)  
- `raw.weather_daily_forecast` → inference dataset (future weather data)  

Function used:
- `create_raw_tables(conn)`

---

###  5. Data Loading

We load data from local files located in `data/raw/`.

####  Historical Data
- Function: `load_raw_historical_data(conn, "../data/raw")`
- Loads files containing `_historical` in filename
- Inserts data into `raw.weather_daily_historical`

####  Forecast Data
- Function: `load_raw_forecast_data(conn, "../data/raw")`
- Loads files containing `_forecast` in filename
- Inserts data into `raw.weather_daily_forecast`


In [4]:
conn.execute("""
SELECT city, COUNT(*) as row_count
FROM raw.weather_daily_historical
GROUP BY city
ORDER BY city;
""").df()

,city,row_count
0,Baku,1827
1,Ganja,1827
2,Nakhchivan,1827
3,Shusha,1827


##  Row Count Validation per City

This query checks how many records (days of weather data) exist for each city in the **historical dataset**.

###  What we did
We grouped the data by `city` and counted the number of rows per city.

###  Result interpretation

- Each city (**Baku, Ganja, Nakhchivan, Shusha**) has **1827 rows**
- This means:
  - Data ingestion was successful
  - All cities have equal time coverage
  - No missing cities or partial loads detected

###  Why this is important
This is a **data integrity check** that ensures:

- Balanced dataset across all cities  
- No ingestion failures  
- Consistent time series length for modeling  

In [5]:
conn.execute("""
SELECT city, COUNT(*) as row_count
FROM raw.weather_daily_forecast
GROUP BY city
ORDER BY city;
""").df()

,city,row_count
0,Baku,7
1,Ganja,7
2,Nakhchivan,7
3,Shusha,7


In [6]:
conn.execute("""
SELECT city,
       MIN(date),
       MAX(date)
FROM raw.weather_daily_historical
GROUP BY city;
""").df()

,city,min(date),max(date)
0,Ganja,2021-04-17 20:00:00,2026-04-17 20:00:00
1,Nakhchivan,2021-04-17 20:00:00,2026-04-17 20:00:00
2,Baku,2021-04-17 20:00:00,2026-04-17 20:00:00
3,Shusha,2021-04-17 20:00:00,2026-04-17 20:00:00


##  Date Range Validation per City

This query checks the **time coverage** of the historical dataset for each city.

###  What we did
We selected the minimum and maximum dates for each city using:

- `MIN(date)` → earliest available record  
- `MAX(date)` → latest available record  
- grouped by `city`

---

###  Result interpretation

For all cities (**Baku, Ganja, Nakhchivan, Shusha**):

- Start date: **2021-04-17**
- End date: **2026-04-17**

This shows that:

- All cities have **consistent time ranges**
- No city is missing historical periods
- The dataset is fully aligned across locations

---

###  Why this is important

This validation ensures:

- Correct time series alignment between cities  
- Reliable input for ML models  
- No gaps in ingestion process at the dataset level  

In [7]:
conn.execute("""
WITH date_series AS (
    SELECT DISTINCT city, DATE(date) as date
    FROM raw.weather_daily_historical
),

expected_dates AS (
    SELECT 
        city,
        DATE(UNNEST(generate_series(
            MIN(date),
            MAX(date),
            INTERVAL 1 DAY
        ))) AS expected_date
    FROM raw.weather_daily_historical
    GROUP BY city
)

SELECT *
FROM expected_dates e
LEFT JOIN date_series d
ON e.city = d.city AND e.expected_date = d.date
WHERE d.date IS NULL;
""").df()

,city,expected_date,city_1,date


##  Missing Dates Check

This query checks whether there are any **missing days in the time series** for each city.

###  What we did
- Generated a full continuous date range using `generate_series`
- Compared it with the actual dates in the dataset
- Selected rows where expected dates do not exist in real data

---

###  Result

- The output is **empty / no meaningful missing matches found**
- This means there are **no missing daily records** in the dataset

---

###  Conclusion

✔ The dataset has a **complete daily time series**  
✔ No gaps detected between min and max dates  
✔ Data is suitable for time-series modeling  

In [8]:
conn.execute("""
SELECT city, date, COUNT(*) as cnt
FROM raw.weather_daily_historical
GROUP BY city, date
HAVING COUNT(*) > 1;
""").df()

,city,date,cnt


##  Duplicate Check

We check if any (city, date) combination appears more than once.

Expected result:
- Each city-date pair must appear exactly once

Duplicates indicate ingestion or merging issues.

No duplicates in our data.

In [9]:
conn.execute("""
PRAGMA table_info('raw.weather_daily_historical');
""").df()

,cid,name,type,notnull,dflt_value,pk
0,0,date,TIMESTAMP,False,None,False
1,1,city,VARCHAR,False,None,False
2,2,weathercode,DOUBLE,False,None,False
3,3,temperature_2m_max,DOUBLE,False,None,False
4,4,temperature_2m_min,DOUBLE,False,None,False
5,5,apparent_temperature_max,DOUBLE,False,None,False
6,6,apparent_temperature_min,DOUBLE,False,None,False
7,7,precipitation_sum,DOUBLE,False,None,False
8,8,precipitation_hours,DOUBLE,False,None,False
9,9,rain_sum,DOUBLE,False,None,False


##  Data Type Validation

We verify whether all columns have correct data types:

- date → TIMESTAMP
- numeric values → FLOAT/DOUBLE
- city → VARCHAR

Incorrect types can break ML models later.

In [10]:
results = {
    "Row Count Check": "PASS",
    "Date Range Check": "PASS",
    "Missing Dates": "PASS",
    "Duplicate Check": "PASS",
    "Data Types": "PASS"
}

pd.DataFrame(results.items(), columns=["Check", "Status"])

,Check,Status
0,Row Count Check,PASS
1,Date Range Check,PASS
2,Missing Dates,PASS
3,Duplicate Check,PASS
4,Data Types,PASS


In [11]:
conn.execute("""
SELECT 
    city,
    EXTRACT(YEAR FROM date) AS year,
    AVG(temperature_2m_max) AS avg_max_temp
FROM raw.weather_daily_historical
GROUP BY city, EXTRACT(YEAR FROM date)
ORDER BY city, year;
""").df()

,city,year,avg_max_temp
0,Baku,2021,23.347683
1,Baku,2022,19.576986
2,Baku,2023,20.253562
3,Baku,2024,19.613934
4,Baku,2025,19.346986
5,Baku,2026,11.013084
6,Ganja,2021,23.451544
7,Ganja,2022,19.691370
8,Ganja,2023,19.687534
9,Ganja,2024,19.058470


##  Average Yearly  Temperature

This query calculates the **average maximum temperature per city per year**.

###  What we did
- Extracted the year from the `date`
- Grouped data by `city` and `year`
- Calculated the average of `temperature_2m_max`

---

###  Result interpretation

- Each row represents a **city-year combination**
- Shows how temperature changes over time for each city
- Example:
  - **Nakhchivan** has the highest temperatures overall
  - **Shusha** is consistently cooler than other cities

---

###  Important note

- Values for **2026 are lower** because the year is incomplete (partial data)

---

###  Why this is useful

✔ Helps understand climate patterns  
✔ Useful for feature engineering (trend, seasonality)  
✔ Supports risk modeling (extreme heat conditions)  

In [12]:
conn.execute("""
SELECT 
    city,
    VARIANCE(precipitation_sum) AS precipitation_variance
FROM raw.weather_daily_historical
GROUP BY city
ORDER BY precipitation_variance DESC
LIMIT 1;
""").df()

,city,precipitation_variance
0,Shusha,23.42377


##  Precipitation Variability Analysis

This query identifies the city with the **highest variability in daily precipitation**.

###  What we did
- Calculated `VARIANCE(precipitation_sum)` for each city  
- Sorted results in descending order  
- Selected the top city with the highest variance  

---

###  Result interpretation

- **Shusha** has the highest precipitation variance (~23.42)
- This means rainfall in Shusha is **more inconsistent and volatile** compared to other cities

---

###  Why this is important

✔ Indicates unstable weather conditions  
✔ Helps detect higher risk for construction delays  
✔ Useful feature for rain-related risk prediction models  v

In [14]:
conn.execute("""
SELECT 
    city,
    date,
    temperature_2m_max
FROM raw.weather_daily_historical
ORDER BY temperature_2m_max DESC
LIMIT 10;
""").df()

,city,date,temperature_2m_max
0,Nakhchivan,2025-08-07 20:00:00,42.35
1,Nakhchivan,2025-08-02 20:00:00,41.95
2,Nakhchivan,2023-07-07 20:00:00,41.70
3,Nakhchivan,2023-08-11 20:00:00,41.45
4,Nakhchivan,2021-07-03 20:00:00,41.40
5,Nakhchivan,2025-08-03 20:00:00,41.40
6,Nakhchivan,2023-08-10 20:00:00,41.15
7,Nakhchivan,2025-07-30 20:00:00,41.05
8,Nakhchivan,2021-07-20 20:00:00,41.00
9,Nakhchivan,2022-07-10 20:00:00,41.00


##  Top 10 Hottest Days

This query retrieves the **top 10 hottest days** across all cities based on maximum temperature.

###  What we did
- Selected `city`, `date`, and `temperature_2m_max`
- Sorted data in descending order by temperature
- Limited results to top 10 records

---

###  Result interpretation

- All top 10 hottest days belong to **Nakhchivan**
- Temperatures exceed **41–42°C**, indicating extreme heat conditions
- These events mostly occur during **summer months (July–August)**

---

###  Why this is important

✔ Identifies extreme weather events  
✔ Useful for temperature risk modeling (e.g., concrete cracking)  
✔ Helps understand which regions face the highest heat stress  

In [ ]:
conn.execute("""
SELECT 
    city,
    EXTRACT(YEAR FROM date) AS year,
    COUNT(*) AS zero_precip_days
FROM raw.weather_daily_historical
WHERE precipitation_sum = 0
GROUP BY city, EXTRACT(YEAR FROM date)
ORDER BY city, year;
""").df()

,city,year,zero_precip_days
0,Baku,2021,189
1,Baku,2022,236
2,Baku,2023,240
3,Baku,2024,227
4,Baku,2025,223
5,Baku,2026,52
6,Ganja,2021,187
7,Ganja,2022,251
8,Ganja,2023,209
9,Ganja,2024,215


##  Days with Zero Precipitation

This query calculates the number of **days without any precipitation** for each city per year.

###  What we did
- Filtered rows where `precipitation_sum = 0`
- Grouped by `city` and `year`
- Counted the number of dry days

---

###  Result interpretation

- **Nakhchivan** has the highest number of dry days → driest region  
- **Shusha** has fewer dry days → more frequent precipitation  
- **2026 values are lower** because the year is incomplete  

---

###  Why this is important

✔ Helps understand dry vs wet climate patterns  
✔ Useful for planning construction activities  
✔ Important feature for rain-related risk modeling  